## Scratchpad

In [1]:
from lcn.core.model import LCN

# Load an example LCN (paths are relative to notebooks/)
filename = "../examples/smokers.lcn"

lcn = LCN()
lcn.from_lcn(file_name=filename)
lcn.summary()

G1 = lcn.build_primal_graph(formula_labels=True)
G2 = lcn.build_structure_graph()
G3 = lcn.simplify_structure_graph()

Parsed LCN format with 12 sentences.
Build the LCN's primal graph.
Build the LCN's structure graph.
Processing sentence: s1: 0.5 <= P(F1 | (F2 and F3)) <= 1.0
Processing sentence: s2: 0.5 <= P(F2 | (F1 and F3)) <= 1.0
Processing sentence: s3: 0.5 <= P(F3 | (F1 and F2)) <= 1.0
Processing sentence: s4: 0.0 <= P((S1 or S2) | F1) <= 0.2 ; True
Processing sentence: s5: 0.0 <= P((S1 or S3) | F3) <= 0.2 ; True
Processing sentence: s6: 0.0 <= P((S2 or S3) | F2) <= 0.2 ; True
Processing sentence: s7: 0.03 <= P(C1 | S1) <= 0.04
Processing sentence: s8: 0.03 <= P(C2 | S2) <= 0.04
Processing sentence: s9: 0.03 <= P(C3 | S3) <= 0.04
Processing sentence: s10: 0.0 <= P(C1 | !S1) <= 0.01
Processing sentence: s11: 0.0 <= P(C2 | !S2) <= 0.01
Processing sentence: s12: 0.0 <= P(C3 | !S3) <= 0.01
Replacing the following bidirected edges with undirected edges: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]
Build the LCN's independence assumptions (LMC).
LCN with 9 atoms and 12 sentences
Atoms: C1, C2, C3, F1, F

In [ ]:
%matplotlib inline
from lcn.core.plotting import plot_graph

plot_graph(G1, layout="kamada_kawai", figsize=(10, 8), title="Primal graph");

In [ ]:
plot_graph(G2, label_attr="label", title="Structure graph");

In [ ]:
plot_graph(G3, label_attr="label", title="Simplified structure graph");

In [5]:
# check if the LCN is a chain graph 
ok = lcn.is_chain_graph()
print(f"Is the LCN a chain graph? {ok}")

Is the LCN a chain graph? True


In [6]:
# get the families of each node in the chain graph
families = lcn.process_chain_graph()
print("Families of each node:")
for family in families:
    node = family["child"]
    print(f"{node}: {family}")

Child: C1, Parents: ['S1-S2-S3'], Scope: ['C1', 'S1', 'S2', 'S3']
Adding sentence s7 with scope {'S1', 'C1'}.
Adding sentence s10 with scope {'S1', 'C1'}.
Child: C2, Parents: ['S1-S2-S3'], Scope: ['C2', 'S1', 'S2', 'S3']
Adding sentence s8 with scope {'S2', 'C2'}.
Adding sentence s11 with scope {'S2', 'C2'}.
Child: C3, Parents: ['S1-S2-S3'], Scope: ['C3', 'S1', 'S2', 'S3']
Adding sentence s9 with scope {'S3', 'C3'}.
Adding sentence s12 with scope {'S3', 'C3'}.
Child: S1-S2-S3, Parents: ['F1-F2-F3'], Scope: ['S1', 'S2', 'S3', 'F1', 'F2', 'F3']
Adding sentence s1 with scope {'F3', 'F2', 'F1'}.
Adding sentence s2 with scope {'F3', 'F2', 'F1'}.
Adding sentence s3 with scope {'F3', 'F2', 'F1'}.
Adding sentence s4 with scope {'S2', 'F1', 'S1'}.
Adding sentence s5 with scope {'S3', 'F3', 'S1'}.
Adding sentence s6 with scope {'S2', 'F2', 'S3'}.
Child: F1-F2-F3, Parents: [], Scope: ['F1', 'F2', 'F3']
Adding sentence s1 with scope {'F3', 'F2', 'F1'}.
Adding sentence s2 with scope {'F3', 'F2', 'F

In [ ]:
# Compile the LCN to a credal network and enumerate the extreme points of each
# local credal set. `method`: "linear" (in-scope-sentence LP, default) or
# "linear-tight" (adds the scope-restricted LMC equalities -- scheme D1).
# `solver`: "ipopt" (local, default) or "scip" (certified global -- scheme D3).
# Set verbosity=2 to stream the ipopt/scip solver progress.
from lcn.inference.marginal.cn.vertices import CredalNetworkVertices

cnv = CredalNetworkVertices.from_lcn(lcn, method="linear", verbosity=1)


In [ ]:
# Credal Variable Elimination now computes ALL posterior marginals in one run():
# it loops the per-target bucket elimination over the network nodes and returns
# {name -> (lower, upper)}. It also records the running times and flags a
# DEGENERATE (all-vacuous [0,1]) solution.
from lcn.inference.marginal.cn.cve import CredalVE

cve = CredalVE(cnv=cnv)
results = cve.run(evidence={}, verbosity=1)

print("\nSingleton-atom marginals P(atom=1):")
for atom in sorted(cve.singleton_marginals):
    lo, hi = cve.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")

print(f"\nbuild time={cve.build_time:.4f}s, "
      f"elimination time={cve.elimination_time:.4f}s, "
      f"total time={cve.total_time:.4f}s")
print(f"degenerate (all marginals vacuous [0,1])? {cve.degenerate}")


In [ ]:
# Tighter / exact options (schemes D4 and D5):
#   CredalVE(coupling="off")          -- bounds the strong extension (default, above);
#   CredalVE(coupling="cross-family") -- D4, forbids vertex combinations that
#                                        violate a cross-family LCN sentence/LMC
#                                        assertion;
#   CredalJT                          -- D5, the junction-tree EXACT NLP (defaults
#                                        to the certified global SCIP backend).
#                                        Exact for both marginal and conditional
#                                        queries. (Formerly CredalVE(coupling="d5").)
from lcn.inference.marginal.cn.cjt import CredalJT

cjt = CredalJT(cnv=cnv)
cjt.run(evidence={}, verbosity=1)

print(f"\nExact marginals via D5 (exact={cjt.d5_exact}):")
for atom in sorted(cjt.singleton_marginals):
    lo, hi = cjt.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")